# Nuisance Parameter Estimation: $f_0$ for Geographical Domain Adaptation

## Overview
This notebook focuses on studying the **geographical transferability** of our **U-TAE** (U-Net with Temporal Attention Encoder) model, specifically designed for **hedgerow mapping**.

The core objective is to estimate the **nuisance parameter** $f_0$ for each spatial location. This parameter accounts for the distribution shift between our source and target domains.

## The $f_0$ Parameter
The nuisance parameter $f_0$ is defined as the density ratio between the target and source distributions:

$$f_0 = \frac{P_{target}}{P_{source}}$$

### Spatio-Temporal Definition
$f_0$ is estimated at the **pixel level** but remains **constant across the temporal dimension** of a given time series. 

## Theoretical Framework
Once estimated, $f_0$ will be used to reweight the U-TAE loss function. This approach follows the **Orthogonal Statistical Learning** framework described in **Paragraph G.2** of:
> *Foster, D. J., & Syrgkanis, V. (2019). Orthogonal Statistical Learning.*

## Estimation Methodology
Estimating $f_0$ requires training a **domain classifier** tasked with distinguishing between source and target pixels. 

1. **Classifier Output**: The model outputs a probability $p(target | x)$, where $x$ represents the input features. This probability reflects the "likeness" or affinity of a pixel to the target domain.
2. **Architecture Choice**: For this task, we employ another **U-TAE** architecture acting as the classifier.
3. **Density Ratio Recovery**: The ratio $f_0$ is subsequently derived from the classifier's output as follows:

$$f_0(x) = \frac{p(target | x)}{p(source | x)} = \frac{p(target | x)}{1 - p(target | x)}$$

In [ ]:
# Imports
import os
import numpy as np
import matplotlib.pyplot as plt

from src.performance_analysis.density_ratio_analyzer import DensityMapGenerator
from f0_estimator import run_nuisance_estimation, tensor_to_df
from src.training.experiments_registry import REGISTRY

In [ ]:
# Experiment configuration
DATASET_ROOT = "/scratch/nathan/data/hedgementation_1.3/hedgementation_1.3"
SAVE_DIR = "results/inference_results"
RUN_INFERENCE = True
NUM_FOLDS = 5

transfer = REGISTRY["nuisance_estimation_temperate"].multi_experiment_factory(NUM_FOLDS)[0]["transfer"]
model_base_dir = "results"
group_idx = transfer["params"]["group_index"]
extra_keys = ["test"]

In [ ]:
# Run cross-validated inference on near_train patches and retrieve the (N, 2, H, W) output tensor
# Canal 0: p0 (probability), Canal 1: f0 (density ratio)

if RUN_INFERENCE:
    generator = run_nuisance_estimation(
        transfer=transfer,
        model_base_dir=model_base_dir,
        dataset_root=DATASET_ROOT,
        extra_keys=extra_keys,
    )
    generator.save(SAVE_DIR)
else:
    generator = DensityMapGenerator.load(SAVE_DIR, DATASET_ROOT)

In [ ]:
# Inspect f0 and p0 distributions across all patches
# Distributions — per fold (train + every extra key)
density_pct_between_a_b = {"a": 0.1, "b": 0.9}

all_split_keys = ["train"] + extra_keys

for fold_idx in range(NUM_FOLDS):
    for key in all_split_keys:
        tensor = generator.results[f"{key}_{fold_idx}"]
        DensityMapGenerator.plot_f0_distribution(
            tensor[:, 1:2],
            title=f"f0 distribution — {key} fold {fold_idx} (far_group: {group_idx})",
            save_path=os.path.join(SAVE_DIR, f"f0_{key}_fold{fold_idx}.png"),
            scale="log",
            abs="$f_0$",
        )
        DensityMapGenerator.plot_f0_distribution(
            tensor[:, 0:1],
            title=f"p0 distribution — {key} fold {fold_idx} (far_group: {group_idx})",
            save_path=os.path.join(SAVE_DIR, f"p0_{key}_fold{fold_idx}.png"),
            abs="$p_0$",
            a=density_pct_between_a_b["a"],
            b=density_pct_between_a_b["b"]
        )

# Distributions — merged train
full_train = generator.results["train"]
DensityMapGenerator.plot_f0_distribution(
    full_train[:, 1:2],
    title=f"f0 distribution — train merged (far_group: {group_idx})",
    save_path=os.path.join(SAVE_DIR, "f0_distribution_train.png"),
    scale="log",
    abs="$f_0$",
)
DensityMapGenerator.plot_f0_distribution(
    full_train[:, 0:1],
    title=f"p0 distribution — train merged (far_group: {group_idx})",
    save_path=os.path.join(SAVE_DIR, "p0_distribution_train.png"),
    abs="$p_0$",
    a=density_pct_between_a_b["a"],
    b=density_pct_between_a_b["b"]
)

In [ ]:
# Diagnostic plots (aggregated over folds, train only)
# Per-fold ambiguity bar chart
p0_median_threshold = 0.3
f0_threshold = 19.0

pct_above_per_fold = []
for fold_idx in range(NUM_FOLDS):
    t = generator.results[f"train_{fold_idx}"]
    per_patch_median = t[:, 0].median(dim=-1).values.median(dim=-1).values  # (n_fold,)
    pct_above_per_fold.append((per_patch_median > p0_median_threshold).float().mean().item() * 100)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(range(NUM_FOLDS), pct_above_per_fold, color="steelblue")
mean_pct = np.mean(pct_above_per_fold)
ax.axhline(mean_pct, color="red", linestyle="--", label=f"Mean: {mean_pct:.1f}%")
ax.set_xlabel("Fold")
ax.set_ylabel(f"% patches (median p0 > {p0_median_threshold})")
ax.set_title(f"Ambiguity per fold — far_group {group_idx}")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, f"fold_ambiguity_bar_{group_idx}.png"))
plt.show()

# Per-patch f0 pixel scatter, coloured by fold
fig, ax = plt.subplots(figsize=(10, 5))
cmap = plt.get_cmap("tab10")
offset = 0
for fold_idx in range(NUM_FOLDS):
    t = generator.results[f"train_{fold_idx}"]
    pct_high = (t[:, 1] > f0_threshold).float().mean(dim=(-2, -1)).numpy() * 100
    xs = np.arange(offset, offset + len(pct_high))
    ax.scatter(xs, pct_high, s=4, color=cmap(fold_idx), label=f"fold {fold_idx}", alpha=0.6)
    offset += len(pct_high)

ax.set_xlabel("Patch index (ordered by fold appearance)")
ax.set_ylabel(f"% pixels with f0 > {f0_threshold}")
ax.set_title(f"Pixel-level far-likeness per patch — far_group {group_idx}")
ax.legend(markerscale=3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, f"fold_scatter_f0_{group_idx}.png"))
plt.show()

In [ ]:
# Heatmap analysis on merged train
F0_THRESHOLD = 19.0
df_train = tensor_to_df(full_train, generator.loaders["train"])

high_f0_pixels = df_train[df_train["f0"] > F0_THRESHOLD]
counts_per_image = high_f0_pixels.groupby("index").size().sort_values(ascending=False)
total_pixels = full_train.shape[2] * full_train.shape[3]
N_patches = full_train.shape[0]
n_affected = len(counts_per_image)

print(f"Patches with at least one f0 > {F0_THRESHOLD} pixel: "
      f"{n_affected} / {N_patches} ({100 * n_affected / N_patches:.1f}%)")

# Patch with most critical pixels
top_index = int(counts_per_image.index[0])
nb_pixels = counts_per_image.iloc[0]
patch_id_top = generator.loaders["train"].dataset.metadata_frame.iloc[top_index]["ID_PATCH"]
f0_top = full_train[top_index, 1].numpy()

print(f"\nImage with most f0 > {F0_THRESHOLD} pixels: index {top_index} <-> ID_PATCH {patch_id_top}")
print(f"Pixels above threshold: {nb_pixels} / {total_pixels} ({100 * nb_pixels / total_pixels:.1f}%)")

generator.display_heatmap(
    split_name="train",
    k=top_index,
    save_path=os.path.join(SAVE_DIR, "f0_heatmap_train_top.png"),
)

generator.display_patches_on_map(
    split_name=f"train",
    patch_indices=[top_index],
    title=f"Image with most f0 > {F0_THRESHOLD}",
)

# Patch closest to 30 % coverage
TARGET_RATIO = 0.30
pct_per_image = counts_per_image / total_pixels
diff_to_target = (pct_per_image - TARGET_RATIO).abs()
candidates = diff_to_target[diff_to_target == diff_to_target.min()].index.tolist()
closest_index = int(np.random.choice(candidates))
closest_patch_id = generator.loaders["train"].dataset.metadata_frame.iloc[closest_index]["ID_PATCH"]
closest_pct = pct_per_image[closest_index] * 100

print(f"\nPatch closest to 30% coverage: index {closest_index} <-> ID_PATCH {closest_patch_id}")
print(f"Pixels above threshold: {counts_per_image[closest_index]} / {total_pixels} ({closest_pct:.1f}%)")

generator.display_heatmap(
    split_name="train",
    k=closest_index,
    save_path=os.path.join(SAVE_DIR, "f0_heatmap_train_30pct.png"),
)

generator.display_patches_on_map(
    split_name=f"train",
    patch_indices=[closest_index],
    title=f"Patch closest to 30% coverage",
)

In [ ]:
# Heatmap on a random patch from each extra-key fold-0
for key in extra_keys:
    split_name = f"{key}_0"
    df_extra = tensor_to_df(generator.results[split_name], generator.loaders[split_name])
    rand_k = int(np.random.choice(df_extra["index"].unique()))

    generator.display_heatmap(
        split_name=split_name,
        k=rand_k,
        save_path=os.path.join(SAVE_DIR, f"f0_heatmap_{key}_fold0.png"),
    )

    generator.display_patches_on_map(
    split_name=split_name,
    patch_indices=[rand_k],
    title=f"Random patch from {split_name}",
)

In [ ]:
FOLD_IDX = 3
P0_TARGET = 0.5
P0_TOL = 0.05
MIN_COVERAGE = 0.20

t = generator.results[f"train_{FOLD_IDX}"]
loader = generator.loaders[f"train_{FOLD_IDX}"]
df_fold = tensor_to_df(t, loader)

total_pixels = t.shape[2] * t.shape[3]

ambiguous_pixels = df_fold[df_fold["p0"].between(P0_TARGET - P0_TOL, P0_TARGET + P0_TOL)]
counts_per_patch = ambiguous_pixels.groupby("index").size()
pct_per_patch = counts_per_patch / total_pixels

ambiguous_indices = pct_per_patch[pct_per_patch > MIN_COVERAGE].index.tolist()

print(f"Retained patches (fold {FOLD_IDX}, >{MIN_COVERAGE*100:.1f}% pixels with p0 ≈ {P0_TARGET} ± {P0_TOL}) : {len(ambiguous_indices)}")
for idx in ambiguous_indices:
    patch_id = loader.dataset.metadata_frame.iloc[idx]["ID_PATCH"]
    print(f"  index {idx} <-> ID_PATCH {patch_id} "
          f"({pct_per_patch[idx]*100:.1f}% pixels with p0 ≈ {P0_TARGET})")

generator.display_patches_on_map(
    split_name=f"train_{FOLD_IDX}",
    patch_indices=ambiguous_indices,
    title=f"Fold {FOLD_IDX} — ambiguous patches",
)

generator.display_patches_on_map(
    split_name=f"train_{FOLD_IDX}",
    patch_indices=ambiguous_indices,
    zoom_patch_idx=ambiguous_indices[0],
    show_rgb=True,
    padding=0.01,
    title=f"Fold {FOLD_IDX} — patch {ambiguous_indices[0]} zoom",
)